In [1]:
import torch
from multires_hash_encoding import MultiresEncodingConfig, MultiresEncoding

In [2]:
from torch import nn

def init_identity(m):
    if isinstance(m, nn.Linear):
        # Fill with identity if possible
        with torch.no_grad():
            m.weight.zero_()
            for i in range(min(m.in_features, m.out_features)):
                m.weight[i, i] = 1.0
            if m.bias is not None:
                m.bias.zero_()

def init_zero(m):
    if isinstance(m, nn.Linear):
        # Fill with identity if possible
        with torch.no_grad():
            m.weight.zero_()
            if m.bias is not None:
                m.bias.zero_()


class Layer(nn.Module):
    def __init__(self, channels_in, channels_out, activation = nn.ReLU(), is_residual = False):
        super().__init__()
        self.channels_in = channels_in
        self.channels_out = channels_out
        self.lin_forward = nn.Linear(self.channels_in, self.channels_out)
        # torch.nn.init.xavier_uniform(self.lin_forward.weight)
        self.activation = activation

        self.is_residual = is_residual

    def forward(self, x):
        if not self.is_residual or True:
            return self.activation(self.lin_forward(x))
        else: 
            return self.activation(self.lin_forward(x)) + x


class MLP(nn.Module):
    """
    Represents the actual neural network part of a model. 
    Does not hold the extra information required to correctly operate it (e.g. input distribtion, domain, etc.)

    
    Attributes
    ---------
    layers : [int]
        Array of ints specifying hidden layers. N entries will result in N-1 layers.
    activation_function : Callable
        Activation function to be used in the network, as of yet not configurable by layer
    """

    def __init__(self,
                 in_features,
                 out_features,
                 hidden_layers, 
                 activation_function = nn.ReLU(),
                 is_residual = False):
        super().__init__()
        if len(hidden_layers) < 1:
            raise Exception("Need at least 2 entries in layer description to create MLP")
                
        self.in_proj = Layer(channels_in=in_features, channels_out=hidden_layers[0], activation=activation_function)
        
        self.net = nn.Sequential(*[
            Layer(layer_in, layer_out, activation_function, is_residual=False) for layer_in, layer_out in zip(hidden_layers, hidden_layers[1:])
        ])

        self.out_proj = nn.Linear(in_features=hidden_layers[-1], out_features=out_features)

        self.is_residual = is_residual
        if self.is_residual:
            self.residual_scaling = nn.Parameter(torch.ones(in_features))

    def forward(self, x):
        if self.is_residual:
            return self.residual_scaling * x + self.out_proj(self.net(self.in_proj(x)))
        else:
            return self.out_proj(self.net(self.in_proj(x)))
    
    def input_shape(self):
        return self.net[0].channels_in
    

# Class taken and adapted from the source code of https://github.com/VincentStimper/normalizing-flows
class ImagePrior(nn.Module):
    """
    Intensities of an image determine probability density of prior
    """

    def __init__(self, image, index_0_1 = False, eps=1.0e-10, dtype=torch.float32, add_flat_gaussian: bool = True):
        """Constructor

        Args:
          image: image as np matrix
          index_0_1: Whether to index from [0, 1]^2 instead of [-1, 1]^2
          eps: small value to add to image to avoid log(0) problems
        """
        super().__init__()

        if add_flat_gaussian:
            xx, yy = torch.meshgrid(torch.linspace(-1, 1, image.shape[0]), torch.linspace(-1, 1, image.shape[1]))
            zz = torch.cat([xx.unsqueeze(2), yy.unsqueeze(2)], 2).view(-1, 2)

            sigma = 10
            dist = torch.distributions.multivariate_normal.MultivariateNormal(torch.zeros(2), torch.eye(2) * sigma)

            log_probs = dist.log_prob(zz).view(*xx.shape)
            probs = torch.exp(log_probs)

            image_tens = torch.tensor(image, dtype=dtype) + probs
        else:
            image_tens = torch.tensor(image, dtype=dtype)

        image_ = (image_tens / image_tens.sum() + eps)[None, None, ..., ...]
        self.image_dims = image_.shape
        self.index_0_1 = index_0_1
        self.register_buffer("log_eps", torch.log(torch.tensor(eps, dtype=dtype)))
        self.register_buffer(
            "density", image_ / torch.sum(image_)
        )

    def log_prob(self, z):
        """
        Args:
          z: value or batch of latent variable

        Returns:
          log probability of the distribution for z
        """

        # Map into [-1, 1] when user wants to index in [0, 1]
        if self.index_0_1:
            z = z * 2 - 1

        z = z[None, None, ..., ...]

        log_probs = torch.log(torch.nn.functional.grid_sample(self.density, z).view(-1))

        return torch.where(torch.isinf(log_probs), self.log_eps, log_probs)
    
    def prob(self, z):
        """
        Args:
          z: value or batch of latent variable

        Returns:
          probability of the distribution for z
        """

        # Map into [-1, 1] when user wants to index in [0, 1]
        if self.index_0_1:
            z = z * 2 - 1

        z = z[None, None, ..., ...]

        return torch.nn.functional.grid_sample(self.density, z).view(-1)

In [3]:
enable_cuda = True
device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')

conf = MultiresEncodingConfig()

conf.nlevels = 8
conf.minres = (16, 16)
conf.maxres = (512, 512)

encoding = MultiresEncoding(**vars(conf), dtype=torch.float32, device = device)

model = MLP(in_features=encoding.encoding_size, out_features=2, hidden_layers=[16, 16, 16], activation_function=nn.SiLU(), is_residual=False)
model.to(device)

MLP(
  (in_proj): Layer(
    (lin_forward): Linear(in_features=16, out_features=16, bias=True)
    (activation): SiLU()
  )
  (net): Sequential(
    (0): Layer(
      (lin_forward): Linear(in_features=16, out_features=16, bias=True)
      (activation): SiLU()
    )
    (1): Layer(
      (lin_forward): Linear(in_features=16, out_features=16, bias=True)
      (activation): SiLU()
    )
  )
  (out_proj): Linear(in_features=16, out_features=2, bias=True)
)

In [5]:
# z = torch.rand((1, 2), device = device) * 2 - 1
z = torch.tensor([[0.1, 0.2], [0.2, 0.3]], dtype=torch.float32, device = device) * 2 - 1

def test_func(x): 
    x = x.unsqueeze(0)  
    enc = encoding(x)
    return model(enc)

jac = torch.func.vmap(torch.func.jacrev(test_func, chunk_size=1))

jacs = jac(z)

GradTrackingTensor(lvl=2, value=
    BatchedTensor(lvl=1, bdim=0, value=
        tensor([[[[1.5000, 3.0000]]],


                [[[3.0000, 4.5000]]]], device='cuda:0')
    )
)
GradTrackingTensor(lvl=2, value=
    BatchedTensor(lvl=1, bdim=0, value=
        tensor([[[[2.5000, 5.0000]]],


                [[[5.0000, 7.5000]]]], device='cuda:0')
    )
)
GradTrackingTensor(lvl=2, value=
    BatchedTensor(lvl=1, bdim=0, value=
        tensor([[[[ 4.2000,  8.4000]]],


                [[[ 8.4000, 12.6000]]]], device='cuda:0')
    )
)
GradTrackingTensor(lvl=2, value=
    BatchedTensor(lvl=1, bdim=0, value=
        tensor([[[[ 7.0000, 14.0000]]],


                [[[14.0000, 21.0000]]]], device='cuda:0')
    )
)
GradTrackingTensor(lvl=2, value=
    BatchedTensor(lvl=1, bdim=0, value=
        tensor([[[[11.5000, 23.0000]]],


                [[[23.0000, 34.5000]]]], device='cuda:0')
    )
)
GradTrackingTensor(lvl=2, value=
    BatchedTensor(lvl=1, bdim=0, value=
        tensor([[[[18.9000, 37